# PubMed Oncology Qwen3.6 27B SFT Fine-Tuning with Unsloth (FP8 LoRA)

**Base Model:** Qwen/Qwen3.6-27B-FP8 (pre-quantized FP8)

**Dataset:** PubMed oncology datagen JSONL — multi-turn QA, continuation, treatment reasoning, beyond-evidence, and self-correction conversations across 11 cancer types

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** This is Phase 1 (SFT). Teaches the model clinical oncology reasoning with thinking chains. Phase 2 (DPO) refines response quality using preference pairs.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order.

## 1. Setup — Configuration, Environment, GPU Check

Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os, sys, subprocess, importlib, importlib.util

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 0: NEUTER GIT                                                         ║
# ║                                                                              ║
# ║  gptqmodel's startup banner shells out to `git rev-parse` to stamp its      ║
# ║  version. /workspace/* is owned by the host user but git runs as root in    ║
# ║  this container → "dubious ownership" failure, which trips up downstream    ║
# ║  code that wasn't expecting a non-zero git exit. We don't use git for       ║
# ║  anything here, so prepend a fake `git` to PATH that always exits 0.        ║
# ║  Must run before any heavy imports (unsloth → transformers → gptqmodel).    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import tempfile, stat
_fake_git_dir = tempfile.mkdtemp(prefix="nogit-")
_fake_git = os.path.join(_fake_git_dir, "git")
with open(_fake_git, "w") as _f:
    _f.write("#!/bin/sh\nexit 0\n")
os.chmod(_fake_git, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
os.environ["PATH"] = _fake_git_dir + os.pathsep + os.environ.get("PATH", "")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ──
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ──
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ──
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR    = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM        = "Qwen/Qwen3.6-27B-FP8"
MODEL_NAME_BASE = "pubmed_oncologist_v2_sft_qwen36_27b_fp8"

# =========================== INPUT DATA ===========================
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl"

# Cap on number of conversations used. 0 = use all loaded conversations.
# Applied AFTER loading the JSONL, BEFORE quality validation. Set e.g. 500
# for a smoke-test run, 3000 for a quick LoRA, leave 0 for full training.
SFT_MAX_EXAMPLES = 3000

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH  = 4096
BATCH_SIZE      = 1
GRAD_ACCUM      = 8        # effective batch = 1 * 8 = 8
LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# =========================== LoRA CONFIGURATION ===========================
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?"

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  Model name:   {MODEL_NAME_BASE}")
print(f"  Input data:   {INPUT_DATA_FILE}")
print(f"  SFT cap:      {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  LoRA output:  {LORA_OUTPUT_DIR}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    FP8 LoRA (pre-quantized FP8 base)")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.8.0.dev0

  unsloth                   2026.5.2             [OK]
  transformers              5.8.0.dev0           [OK]
  trl                       0.24.0               [OK]
  causal_conv1d             0.0.local            [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   Qwen/Qwen3.6-27B-FP8
  Model name:   pubmed_oncologist_v2_sft_qwen36_27b_fp8
  Input data:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  SFT cap:      3000
  LoRA output:  /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_fp8/lora_adapters
  LoRA:         r=32, alpha=32, targets=7 modu

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from datagen.

- 33,349 conversations across 11 cancer types
- Data types: QA (with thinking), continuation, treatment reasoning, beyond-evidence, self-correction
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`
- System prompts are extracted from the JSONL at load time (stays in sync with datagen)

In [2]:
import json, os
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

raw_conversations = []
cancer_by_index = []          # parallel to raw_conversations — needed for stratified cap
cancer_type_counts = defaultdict(int)
data_type_counts = defaultdict(int)
system_prompts_by_cancer = {}  # cancer_type -> system prompt text

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)

        cancer = conv.get("cancer_type", "unknown")
        dtype = conv.get("data_type", "unknown")
        cancer_type_counts[cancer] += 1
        data_type_counts[dtype] += 1

        # Extract system prompt per cancer type
        sys_msg = conv["conversations"][0]["value"]
        if cancer not in system_prompts_by_cancer:
            system_prompts_by_cancer[cancer] = sys_msg

        # Only keep conversations key — strip data_type/cancer_type
        # so standardize_sharegpt doesn't choke on non-string values
        raw_conversations.append({"conversations": conv["conversations"]})
        cancer_by_index.append(cancer)

dataset = HFDataset.from_list(raw_conversations)

# Apply SFT cap from config — STRATIFIED per cancer type, deterministic seed.
# Splits SFT_MAX_EXAMPLES evenly across all cancer types (remainder goes to the
# types with the most data). If a type has fewer rows than its quota, all are
# taken and the total comes in slightly under the cap.
if SFT_MAX_EXAMPLES:
    import random
    random.seed(42)
    if SFT_MAX_EXAMPLES < 0:
        raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")
    if len(dataset) > SFT_MAX_EXAMPLES:
        cancers_sorted = sorted(cancer_type_counts.keys(), key=lambda c: -cancer_type_counts[c])
        n_types = len(cancers_sorted)
        base = SFT_MAX_EXAMPLES // n_types
        extras = SFT_MAX_EXAMPLES - base * n_types
        target = {c: base + (1 if i < extras else 0) for i, c in enumerate(cancers_sorted)}

        idx_by_cancer = defaultdict(list)
        for i, c in enumerate(cancer_by_index):
            idx_by_cancer[c].append(i)

        selected_idx = []
        per_type_taken = {}
        for c in cancers_sorted:
            pool = idx_by_cancer[c]
            random.shuffle(pool)
            take = min(len(pool), target[c])
            per_type_taken[c] = take
            selected_idx.extend(pool[:take])
        random.shuffle(selected_idx)

        print(f"  Stratified cap: target {SFT_MAX_EXAMPLES} across {n_types} cancer types "
              f"(~{base}/type, +1 for top {extras})")
        print(f"  Actually selected: {len(selected_idx)} rows")
        for c in cancers_sorted:
            short = "(undersized)" if per_type_taken[c] < target[c] else ""
            print(f"    {c:30s} {per_type_taken[c]:>5d} / {target[c]:<5d} {short}")

        dataset = dataset.select(selected_idx)
        # Rebuild cancer_type_counts so the breakdown below reflects the capped data
        cancer_type_counts = defaultdict(int, per_type_taken)
    else:
        print(f"  SFT_MAX_EXAMPLES={SFT_MAX_EXAMPLES}, but only {len(dataset)} examples available; using all")

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations")
print(f"Cancer types: {len(cancer_type_counts)}")
print(f"Data types: {len(data_type_counts)}")
print(f"Unique system prompts: {len(system_prompts_by_cancer)}")
print(f"Columns: {dataset.column_names}")

print(f"\nPer-cancer breakdown:")
for ct, c in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ct:30s} {c:>5d} conversations")

print(f"\nPer-data-type breakdown:")
for dt, c in sorted(data_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {dt:25s} {c:>5d} conversations")


LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  Capping SFT dataset from 33349 to 3000 examples (shuffled, seed=42)

Total dataset: 3000 multi-turn conversations
Cancer types: 11
Data types: 5
Unique system prompts: 11
Columns: ['conversations']

Per-cancer breakdown:
  multi                           5002 conversations
  pubmed_brain_tumour             3324 conversations
  pubmed_bone_cancer              3231 conversations
  pubmed_colon_cancer             3004 conversations
  pubmed_breast_cancer            2974 conversations
  pubmed_ovarian_cancer           2927 conversations
  pubmed_lung_cancer              2683 conversations
  pubmed_kidney_cancer            2629 conversations
  pubmed_gastric_cancer           2598 conversations
  pubmed_skin_cancer              2597 conversations
  pubmed_prostate_cancer          2380 conversations

Per-data-type breakdown:
  qa                

## 3. Validate & Summarize Dataset

Verify data quality: turn structure, non-empty responses, thinking tag presence.

In [3]:
from collections import Counter

bad_examples = []
empty_responses = []
think_count = 0
no_think_count = 0

for i, example in enumerate(dataset):
    convs = example["conversations"]
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count >= 3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    has_think = any(c["value"].strip().startswith("<think>") for c in convs if c["from"] == "gpt")
    if has_think:
        think_count += 1
    else:
        no_think_count += 1

turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  With <think> tags: {think_count} ({100*think_count/len(dataset):.1f}%)")
print(f"  Without <think>:   {no_think_count} ({100*no_think_count/len(dataset):.1f}%)")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n  Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n  Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

print(f"\nCANCER TYPE DISTRIBUTION:")
max_name_len = max(len(n) for n in cancer_type_counts)
for name, count in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    bar = "#" * (count // 200)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(cancer_type_counts.values()):>5}")

print(f"\nDataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 3000
  Bad structure: 0
  Empty responses: 0
  With <think> tags: 1974 (65.8%)
  Without <think>:   1026 (34.2%)
  Turn distribution: {3: 2185, 5: 523, 7: 66, 9: 226}

CANCER TYPE DISTRIBUTION:
  multi                   5002  #########################
  pubmed_brain_tumour     3324  ################
  pubmed_bone_cancer      3231  ################
  pubmed_colon_cancer     3004  ###############
  pubmed_breast_cancer    2974  ##############
  pubmed_ovarian_cancer   2927  ##############
  pubmed_lung_cancer      2683  #############
  pubmed_kidney_cancer    2629  #############
  pubmed_gastric_cancer   2598  ############
  pubmed_skin_cancer      2597  ############
  pubmed_prostate_cancer  2380  ###########
  TOTAL                  33349

Dataset validated and ready for training


## 4. Load Model & Tokenizer (FP8)

In [4]:
import re
import torch
from unsloth import FastLanguageModel

# ── Workaround for a bug in Qwen3.6-27B-FP8's config.json ──────────────────────
# `modules_to_not_convert` contains 65 entries shaped like
# 'model.language_model.layers.N.mlp.gate' — referencing a `.gate` module that
# does not exist in this checkpoint. transformers' `should_convert_module` uses
# `full_name.endswith(key)`, which greedily matches `gate_proj` against `gate`
# and skips FP8Linear conversion for every language MLP gate_proj. The FP8
# weights then load into a bf16 nn.Linear with no scale (weight_scale_inv shows
# up as UNEXPECTED in the load report), silently corrupting gate_proj outputs.
#
# Fix: monkey-patch the matcher to require dot-separated suffix matching, so a
# pattern of `...mlp.gate` no longer matches `...mlp.gate_proj`.
from transformers.integrations import finegrained_fp8 as _fp8

def _strict_should_convert_module(full_name, patterns=None):
    if patterns is None:
        return True
    should_not_convert = any(
        re.match(f"{key}\\.", full_name)
        or full_name == key
        or full_name.endswith(f".{key}")
        for key in patterns
    )
    return not should_not_convert

_fp8.should_convert_module = _strict_should_convert_module

# ── Workaround for peft<>gptqmodel version drift ──────────────────────────────
# peft/tuners/lora/awq.py:98 does `from gptqmodel.nn_modules.qlinear.gemm_awq
# import AwqGEMMQuantLinear` whenever gptqmodel is installed. The installed
# gptqmodel version no longer exports that class, so PEFT's LoRA dispatcher
# crashes at adapter-injection time — even though this model is FP8 (never AWQ)
# and the subsequent isinstance() check would be False. Inject a stub class so
# the import succeeds; isinstance() against the stub is always False.
try:
    from gptqmodel.nn_modules.qlinear import gemm_awq as _gptq_awq
    if not hasattr(_gptq_awq, "AwqGEMMQuantLinear"):
        class _AwqGEMMQuantLinearStub:
            pass
        _gptq_awq.AwqGEMMQuantLinear = _AwqGEMMQuantLinearStub
except ImportError:
    pass
# ───────────────────────────────────────────────────────────────────────────────

# FP8 model is pre-quantized with FineGrainedFP8Config baked into config.json.
# Must explicitly pass load_in_4bit=False — Unsloth's default is True, which
# would inject a BitsAndBytesConfig and collide with the model's FP8 config.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)

# Qwen3.6-27B is a VLM — Unsloth returns a Qwen3VLProcessor (multimodal wrapper),
# not a raw tokenizer. For text-only SFT, extract the inner tokenizer so that
# chat template / SFTTrainer get a real PreTrainedTokenizer.
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from processor — text-only SFT mode)")

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

# Sanity check: confirm the gate_proj layers got the FP8Linear swap.
gp_class = type(model.model.language_model.layers[0].mlp.gate_proj).__name__
print(f"  gate_proj class: {gp_class}  (expect FP8Linear)")
assert "FP8" in gp_class, (
    f"gate_proj is {gp_class}, not FP8Linear — the modules_to_not_convert "
    "patch failed; gate_proj outputs would be silently corrupted."
)

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: FP8 LoRA (pre-quantized FP8 base)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+
Transformers : 5.8.0.dev0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.8.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1584 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Format Dataset for Chat Template

Map ShareGPT roles to chat template roles, apply Qwen3's native chat template (ChatML), then manual sequence packing for 100% token utilization.

**Note:** `enable_thinking=False` is used in `apply_chat_template` because the training data already contains `<think>` blocks in the GPT response text. Without this flag, the template would inject a second `<think>` tag.


In [ ]:
from datasets import Dataset as HFDataset

# Map ShareGPT roles to chat template roles directly
ROLE_MAP = {"system": "system", "human": "user", "gpt": "assistant"}

chat_conversations = []
for example in dataset:
    messages = []
    for turn in example["conversations"]:
        messages.append({
            "role": ROLE_MAP[turn["from"]],
            "content": turn["value"],
        })
    chat_conversations.append(messages)

# enable_thinking=False because data already has <think> blocks in responses
formatted_texts = tokenizer.apply_chat_template(
    chat_conversations,
    tokenize=False,
    enable_thinking=False,
)

# ── Manual sequence packing ──
eos_id = tokenizer.eos_token_id
num_conversations = 0
all_ids = []

for text in formatted_texts:
    if not text:
        continue
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids.extend(ids)
    all_ids.append(eos_id)
    num_conversations += 1

total_tokens = len(all_ids)
num_chunks = total_tokens // MAX_SEQ_LENGTH
all_ids = all_ids[:num_chunks * MAX_SEQ_LENGTH]
chunks = [all_ids[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(num_chunks)]

packed_texts = tokenizer.batch_decode(chunks, skip_special_tokens=False)
dataset = HFDataset.from_dict({"text": packed_texts})
dataset = dataset.shuffle(seed=42)

wasted = total_tokens - (num_chunks * MAX_SEQ_LENGTH)
print(f"Dataset packed: {num_conversations} conversations -> {num_chunks} chunks of {MAX_SEQ_LENGTH} tokens")
print(f"  Total tokens: {total_tokens:,}  |  Wasted (tail): {wasted:,} ({100*wasted/total_tokens:.1f}%)")
print(f"  Token utilization: ~100% (no padding)")
print(f"\n--- Sample packed text (first 500 chars) ---")
print(dataset[0]["text"][:500])


## 6. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"  Target modules: {LORA_TARGET_MODULES}")

## 7. Trainer Setup

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        report_to="none",
        dataset_num_proc=1,
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"Trainer configured (PubMed Oncology Qwen3.6 27B FP8 LoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: manual (pre-packed, each example = {MAX_SEQ_LENGTH} tokens, zero padding)")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"  Dataset: {len(dataset)} packed chunks")

## 8. Train

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

## 9. Save LoRA Adapters

Save the trained LoRA adapters and system prompts. The DPO notebook (Phase 2) expects the LoRA at this path.

In [ ]:
import json
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(system_prompts_by_cancer, f, indent=2)

print(f"\nLoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(system_prompts_by_cancer)} cancer types)")
print(f"  DPO notebook expects LoRA at: {{PROJECT_ROOT}}/output/pubmed_oncologist_v2_sft/lora_adapters")

for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

## 10. Test Inference

Smoke test with oncology questions across different cancer types. This is a thinking model — responses will include `<think>` reasoning blocks.

In [ ]:
import warnings, logging, json, re
from transformers import TextStreamer

# Suppress transformers deprecation warning bug (msg % FutureWarning formatting error)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

# Re-apply the gate_proj FP8 swap fix in case this cell runs in a fresh kernel.
# See cell 4 for explanation. Idempotent.
from transformers.integrations import finegrained_fp8 as _fp8
def _strict_should_convert_module(full_name, patterns=None):
    if patterns is None:
        return True
    return not any(
        re.match(f"{key}\\.", full_name) or full_name == key or full_name.endswith(f".{key}")
        for key in patterns
    )
_fp8.should_convert_module = _strict_should_convert_module

# Load model from saved adapters if not already in memory
if "model" not in dir() or model is None:
    from unsloth import FastLanguageModel
    print(f"Model not in memory — loading from saved adapters: {LORA_OUTPUT_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=LORA_OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )
    # Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer

if "system_prompts_by_cancer" not in dir() or not system_prompts_by_cancer:
    with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
        system_prompts_by_cancer = json.load(f)

FastLanguageModel.for_inference(model)

test_cancers = list(system_prompts_by_cancer.keys())[:3]

print(f"INFERENCE TEST — {len(test_cancers)} CANCER TYPES\n")

for cancer_type in test_cancers:
    system_prompt = system_prompts_by_cancer[cancer_type]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # Thinking model: let Qwen3 default (enable_thinking=True) inject <think>
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  CANCER TYPE: {cancer_type.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability. Also verifies the DPO notebook can load these adapters.

In [ ]:
import gc, torch, json, re
from pathlib import Path

del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Re-apply the gate_proj FP8 swap fix (idempotent) — see cell 4.
from transformers.integrations import finegrained_fp8 as _fp8
def _strict_should_convert_module(full_name, patterns=None):
    if patterns is None:
        return True
    return not any(
        re.match(f"{key}\\.", full_name) or full_name == key or full_name.endswith(f".{key}")
        for key in patterns
    )
_fp8.should_convert_module = _strict_should_convert_module

from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
# Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

FastLanguageModel.for_inference(model2)

with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_cancer = list(reloaded_prompts.keys())[0]
test_system = reloaded_prompts[test_cancer]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (cancer type: {test_cancer}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk.")
print(f"DPO notebook (Phase 2) can now load this LoRA from: {LORA_OUTPUT_DIR}")

print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()